# LipiLens V1 - Data Pipeline

This notebook prepares the Modi Lipi dataset for PyTorch training.

We will:

- Load the existing Training and Testing directories
- Define image transformations
- Create a PyTorch Dataset
- Split Training into train and validation sets
- Create DataLoaders
- Verify the resulting tensors

The official Testing set will remain untouched for final evaluation.

In [1]:
# Import the libraries needed for the data pipeline

from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

## 1. Dataset Paths

The dataset already contains separate `Training` and `Testing`
directories with 46 character classes.

We keep the official Testing set completely untouched.

In [2]:
# Project root
project_root = Path.cwd().parent

# Dataset directories
dataset_path = project_root / "MODI MATRA DATASET" / "MODI MATRA DATASET"

train_path = dataset_path / "Training"
test_path = dataset_path / "Testing"

print("Training path:", train_path)
print("Testing path:", test_path)

Training path: d:\Projects\Lipilens\MODI MATRA DATASET\MODI MATRA DATASET\Training
Testing path: d:\Projects\Lipilens\MODI MATRA DATASET\MODI MATRA DATASET\Testing


## 2. Image Transformations

The CNN will receive grayscale images of size `32 × 32`.

The pipeline:

1. Convert to grayscale
2. Resize to 32 × 32
3. Convert to a PyTorch tensor
4. Normalize pixel values

In [3]:
# Image preprocessing used by the CNN
# WHY ? 
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

## 3. Create the PyTorch Dataset

Our dataset follows this structure:

Training/
    a/
    aa/
    ah/
    ...
    
Each folder represents one Modi character class.

The Dataset converts:

image file → image tensor + class ID

In [4]:
class ModiDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        # Sort classes so class IDs remain consistent
        self.classes = sorted([
            folder.name
            for folder in self.root_dir.iterdir()
            if folder.is_dir()
        ])

        # Map class name → integer ID
        self.class_to_idx = {
            class_name: idx
            for idx, class_name in enumerate(self.classes)
        }

        # Store every image path and its label
        self.samples = []

        for class_name in self.classes:

            class_dir = self.root_dir / class_name

            for image_path in sorted(class_dir.iterdir()):

                if image_path.is_file():
                    self.samples.append((
                        image_path,
                        self.class_to_idx[class_name]
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        # Open image
        image = Image.open(image_path).convert("L")

        # Apply preprocessing
        if self.transform:
            image = self.transform(image)

        return image, label

## 4. Load the Dataset

We now create Dataset objects for the official Training and
Testing directories.

The Training dataset will later be divided into:

- Training
- Validation

The Testing dataset will remain untouched.

In [5]:
# Create dataset objects

full_train_dataset = ModiDataset(
    train_path,
    transform=transform
)

test_dataset = ModiDataset(
    test_path,
    transform=transform
)

print("Training samples:", len(full_train_dataset))
print("Testing samples:", len(test_dataset))
print("Number of classes:", len(full_train_dataset.classes))

Training samples: 19320
Testing samples: 8280
Number of classes: 46


## 5. Train / Validation Split

The official Training set contains 19,320 images.

We reserve 15% of it for validation.

The official Testing set is not touched.

This gives us:

- ~85% Training
- ~15% Validation
- Official Testing set for final evaluation

In [6]:
# Make the split reproducible

generator = torch.Generator().manual_seed(42)

train_size = int(0.85 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=generator
)

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

Training: 16422
Validation: 2898
Testing: 8280


## 6. Create DataLoaders

DataLoaders group individual samples into batches.

The CNN will receive tensors in the form:

`[batch_size, channels, height, width]`

We use a batch size of 64.

Training data is shuffled, while validation and testing are not.

In [7]:
# Batch size used during CNN training
BATCH_SIZE = 64

# Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("DataLoaders ready.")

DataLoaders ready.


## 7. Verify a Batch

Before building the CNN, we verify exactly what PyTorch is
giving the model.

Expected image shape:

`[64, 1, 32, 32]`

Expected label shape:

`[64]`

In [8]:
# Get one batch from the training DataLoader

images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

Images: torch.Size([64, 1, 32, 32])
Labels: torch.Size([64])
Image dtype: torch.float32
Label dtype: torch.int64


Training - Data model learns from
Validation - used to check how well model is learning during development. Model does not update its weight using validation. 
Test - To test model. 

### Why Is the Image Shape [64, 1, 32, 32]?
# [64, 1, 32, 32]
# [ batch, channels, height, width ]
- channels - here 1 (grayscale) 3 if RGB
- 32 image height
- 32 image width
- batch size 64 - pytorch gives 64 images to CNN at a time
